In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv
import os

load_dotenv()

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
#python -m venv APICalls
#.\APICalls\Scripts\activate

In [3]:
# Step 2: Define FRED Series IDs for Rent CPI by Region
series_ids = {
    "US City Average": "CUUR0000SEHA",
    "Northeast": "CUUR0100SEHA",
    "Midwest": "CUUR0200SEHA",
    "South": "CUUR0300SEHA",
    "West": "CUUR0400SEHA"
}

# Step 3: Define Date Range (last 12 months)
end_date = datetime.today()
start_date = end_date - timedelta(days=365)

In [ ]:
# --- Query Fred to pull regional rent averages (12 months, monthly)

def fetch_fred_data(series_id):
    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id,
        "api_key": os.getenv("FRED_API_KEY"),
        "file_type": "json",
        "observation_start": start_date.strftime('%Y-%m-%d'),
        "observation_end": end_date.strftime('%Y-%m-%d'),
    }
    response = requests.get(url, params=params, verify=False)
    response.raise_for_status()  # Raise error if request fails
    data = response.json()
    df = pd.DataFrame(data["observations"])
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df[["date", "value"]].rename(columns={"value": series_id})

# Step 5: Loop through each region and fetch data
dfs = []
for name, series_id in series_ids.items():
    df = fetch_fred_data(series_id)
    df = df.rename(columns={series_id: name})
    dfs.append(df)

# Step 6: Merge all regional data into one DataFrame
df_merged = dfs[0]
for df in dfs[1:]:
    df_merged = df_merged.merge(df, on="date", how="outer")

# Step 7: Preview the result
df_merged = df_merged.sort_values("date").reset_index(drop=True)
print(df_merged.head(12))  # Print the last 12 months

# Optional: Export to Excel
df_merged.to_excel("regional_rent_cpi_fred.xlsx", index=False)

         date  US City Average  Northeast  Midwest    South     West
0  2024-04-01          416.386    431.822  350.438  389.000  455.275
1  2024-05-01          417.772    433.802  351.900  390.113  456.512
2  2024-06-01          418.820    436.003  353.156  390.903  456.998
3  2024-07-01          420.577    438.034  354.887  392.174  459.088
4  2024-08-01          422.223    441.048  356.635  393.450  460.199
5  2024-09-01          423.821    443.231  358.075  394.789  461.749
6  2024-10-01          425.381    445.347  360.659  395.963  462.823
7  2024-11-01          426.651    447.338  362.342  396.806  463.879
8  2024-12-01          428.151    449.357  363.549  397.871  465.647
9  2025-01-01          429.506    451.349  364.642  399.187  466.701
10 2025-02-01          430.603    452.655  366.602  399.556  468.076
11 2025-03-01          431.798    454.054  367.844  400.242  469.698


In [5]:
# --- Excess Cleanup
del dfs, end_date, name, series_id, start_date, df, series_ids

In [6]:
with open("C:/Users/ben.pharris/Documents/GitHub/API_Calls/APICalls/rents.txt", "r") as file:
    lines = file.readlines()

display(file)

<_io.TextIOWrapper name='C:/Users/ben.pharris/Documents/GitHub/API_Calls/APICalls/rents.txt' mode='r' encoding='cp1252'>

In [7]:
# --- Creating column structure
import re

columns = re.findall(r'\d{4}', lines[0])
columns.insert(0, "State")

In [8]:
# Parsing rows
rows = [] 

for line in lines[1:]:
    match = re.match(r'^(.*?)(?=\s+\$?\d)', line)
    if match:
        state = match.group(1).strip()
        values = re.findall(r'(?:NA|\$\d+)', line)
        cleaned_values = [float(v.replace('$', '')) if v != 'NA' else None for v in values]
        row = [state] + cleaned_values
        rows.append(row)

In [9]:
rent = pd.DataFrame(rows, columns=columns)

del cleaned_values, columns, file, line, lines, match, row, rows, state, values 

In [ ]:
    # --- Final Cleaning
rent['1983'] = 0.3 * rent["1990"] + 0.7 * rent["1980"]
rent = rent[["State", "1983"]]

In [12]:
state_to_region = {
    # Northeast
    'Connecticut': 'Northeast', 'Maine': 'Northeast', 'Massachusetts': 'Northeast', 'New Hampshire': 'Northeast',
    'Rhode Island': 'Northeast', 'Vermont': 'Northeast', 'New Jersey': 'Northeast', 'New York': 'Northeast',
    'Pennsylvania': 'Northeast',

    # Midwest
    'Illinois': 'Midwest', 'Indiana': 'Midwest', 'Iowa': 'Midwest', 'Kansas': 'Midwest', 'Michigan': 'Midwest',
    'Minnesota': 'Midwest', 'Missouri': 'Midwest', 'Nebraska': 'Midwest', 'North Dakota': 'Midwest',
    'Ohio': 'Midwest', 'South Dakota': 'Midwest', 'Wisconsin': 'Midwest',

    # South
    'Alabama': 'South', 'Arkansas': 'South', 'Delaware': 'South', 'Florida': 'South', 'Georgia': 'South',
    'Kentucky': 'South', 'Louisiana': 'South', 'Maryland': 'South', 'Mississippi': 'South', 'North Carolina': 'South',
    'Oklahoma': 'South', 'South Carolina': 'South', 'Tennessee': 'South', 'Texas': 'South', 'Virginia': 'South',
    'West Virginia': 'South', 'District of Columbia': 'South',

    # West
    'Alaska': 'West', 'Arizona': 'West', 'California': 'West', 'Colorado': 'West', 'Hawaii': 'West',
    'Idaho': 'West', 'Montana': 'West', 'Nevada': 'West', 'New Mexico': 'West', 'Oregon': 'West',
    'Utah': 'West', 'Washington': 'West', 'Wyoming': 'West',

    #US City Average
    'United States' : 'US City Average'
}

rent["Region"] = rent["State"].map(state_to_region)

regional_rent = rent.groupby("Region")["1983"].mean().reset_index()

In [ ]:
# --- CHATGPT merging base rents and FRED values and generating absolute rents for last 12 months

# 0) Inspect your column names
print("df_merged columns:", df_merged.columns.tolist())
print("regional_rent columns:", regional_rent.columns.tolist())

# 1) Strip any stray whitespace from column names (just in case)
df_merged.columns = df_merged.columns.str.strip()
regional_rent.columns = regional_rent.columns.str.strip()

# 2) Grab the two columns in regional_rent dynamically
#    (we assume the first is the region name, second is the 1983 rent)
region_col, base_col = regional_rent.columns[:2]

# 3) Rename them to our “region” and “base_rent_1983”
regional_rent = regional_rent.rename(columns={
    region_col: 'region',
    base_col:   'base_rent_1983'
})

# 4) Melt your FRED table from wide → long
#    (this creates a “region” column and a “rent_index” column)
df_long = df_merged.melt(
    id_vars='date',
    var_name='region',
    value_name='rent_index'
)

# 5) Merge in the base‐year rents
df_long = df_long.merge(
    regional_rent,
    on='region',
    how='left',          # you can also use 'inner' if every region must match
    validate='many_to_one'
)

# 6) Scale the index back to dollars
df_long['absolute_rent'] = df_long['rent_index'] / 100 * df_long['base_rent_1983']

# 7) (Optional) Pivot back to wide if you want one column per region again
df_wide = (
    df_long
    .pivot(index='date', columns='region', values='absolute_rent')
    .reset_index()
)

df_merged columns: ['date', 'US City Average', 'Northeast', 'Midwest', 'South', 'West']
regional_rent columns: ['Region', 'base_rent_1983']


In [17]:
del base_col, df_merged, region_col, regional_rent, rent, state_to_region

In [19]:
df_wide = df_wide.round(2)

df_wide.to_csv('C:/Users/ben.pharris/Documents/GitHub/API_Calls/APICalls/rents.csv', index=False)